In [1]:
# single cell libraries
import scanpy as sc
import anndata as ad
import scrublet as scr
import leidenalg, igraph
import bbknn

# processing tools
import numpy as np
import pandas as pd
import scipy
import itertools
from fa2 import ForceAtlas2

# plotting tools
import matplotlib.pyplot as plt

# file grab
from pathlib import Path
import pooch
import hashlib

# sc settings
sc.settings.verbosity = 1
sc.logging.print_versions()
results_file = "./write/_"

sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")

/root/miniconda/envs/unified_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# restore working adata
adata=sc.read_h5ad("cache/adata_pbmc68k.h5ad")
adata

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'cache/adata_pbmc68k.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [6]:
# mask adata with t-like mask
t_like = adata.obs['leiden_r1.0'].isin(['0','1','2','3','4','5','6'])
adata.obs['is_t_like'] = t_like  # persist this tag
adata.write('pbmc_with_tmask.h5ad')
adata_t = adata[adata.obs['is_t_like']].copy()

In [9]:
# re-pp t-like mask to eliminate NK-like cells

sc.pp.filter_genes(adata_t, min_cells=20)
sc.pp.highly_variable_genes(adata_t, flavor="seurat")
sc.pp.pca(adata_t, svd_solver='arpack')
sc.tl.diffmap(adata_t)
sc.pp.neighbors(adata_t, n_neighbors=10, n_pcs=40, metric="X_diffmap")
sc.tl.umap(adata_t)
sc.pl.umap(
    adata_t,
    size=4, alpha=0.8, legend_loc="on data", frameon=False,
    save="_pbmc_with_tmask_1_unlabeled.png"
)
adata_t

ValueError: Metric is neither callable, nor a recognised string

In [10]:
adata_t.write('pbmc_with_tmask.h5ad')